In [1]:
# Importing Libraries
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\ankit\AppData\Local\Temp\ipykernel_1260\832470481.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase
C:\Users\ankit\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Connect MySQL database
from urllib.parse import quote_plus

host = "localhost"
port = "3306"
username = "root"
password = quote_plus("MySQL@2012")
database_schema = "text_to_sql"

mysql_uri = (
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"
)

db = SQLDatabase.from_uri(
    mysql_uri,
    sample_rows_in_table_info=2
)

In [5]:
db = SQLDatabase.from_uri(mysql_uri,sample_rows_in_table_info=1)

db.get_table_info()

'\nCREATE TABLE `2017_budgets` (\n\t`Product Name` TEXT, \n\t`2017 Budgets` DOUBLE\n)COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB\n\n/*\n1 rows from 2017_budgets table:\nProduct Name\t2017 Budgets\nProduct 1\t3016489.2089999998\n*/\n\n\nCREATE TABLE customers (\n\t`Customer Index` INTEGER, \n\t`Customer Names` TEXT\n)COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB\n\n/*\n1 rows from customers table:\nCustomer Index\tCustomer Names\n1\tGeiss Company\n*/\n\n\nCREATE TABLE products (\n\t`Index` INTEGER, \n\t`Product Name` TEXT\n)COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB\n\n/*\n1 rows from products table:\nIndex\tProduct Name\n1\tProduct 1\n*/\n\n\nCREATE TABLE regions (\n\tid INTEGER, \n\tname TEXT, \n\tcounty TEXT, \n\tstate_code TEXT, \n\tstate TEXT, \n\ttype TEXT, \n\tlatitude DOUBLE, \n\tlongitude DOUBLE, \n\tarea_code INTEGER, \n\tpopulation INTEGER, \n\thouseholds INTEGER, \n\tmedian_income INTEGER, \n\tland_area INTEGER, \

In [6]:
# Create the LLM Prompt Template
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else. Provide me sql query in a single line dont add line breaks
Table Schema: {schema}
Question: {question}
SQL Query:
"""

prompt = ChatPromptTemplate.from_template(template)

In [7]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key = 'ABC')

In [30]:
sql_chain = (
    RunnablePassthrough.assign(
        schema=lambda _: get_schema(db)
    )
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [32]:
# test the SQL query chain with a sample question
resp = sql_chain.invoke({"question": "What is the total 'Line Total' for Geiss Company"})
print(resp)

```sql
SELECT SUM(Line Total) FROM sales_orders_old WHERE `Customer Name Index` = (SELECT `Customer Index` FROM customers WHERE `Customer Names` = 'Geiss Company');
```


In [33]:
resp = sql_chain.invoke({"question": "tell me region wise product"})
print(resp)

```sql
SELECT r.name AS Region, p.`Product Name` AS Product FROM regions r JOIN products p ON r.id = p.`Index`;
```


In [34]:
resp = sql_chain.invoke({"question": "how much total no. of product"})
print(resp)

SELECT COUNT(*) AS total_products FROM products;
